# Лабораторная работа №6
## Apache Spark: обработка данных в памяти, потоковая обработка и визуализация

**Вариант 1 — интернет-магазин одежды.** Среда: Python, PySpark, Google Colab, Power BI.

**Цель:** изучить SparkSession, RDD, DataFrame, Spark SQL, кэширование, Structured Streaming и подготовить результаты для визуализации.

Время выполнения зависит от ресурсов Google Colab. В задании 5 используй фактические замеры, не подставляй вымышленные значения.

## Задание 1. Установка и запуск Spark

In [ ]:
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null 2>&1
!pip install -q pyspark==4.2.0

In [ ]:
import os, sys, time, math
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType

spark = (SparkSession.builder.appName("LR6_Spark_Variant1")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext
print("Версия Spark:", spark.version)
print("Режим:", sc.master)
print("Параллелизм по умолчанию:", sc.defaultParallelism)

**Ответы к заданию 1.** Driver выполняет пользовательскую программу, строит план и координирует задачи. Executor исполняет задачи и может кэшировать данные. `local[*]` использует доступные ядра текущей виртуальной машины. Число одновременно выполняемых задач ограничено доступными ядрами/параллелизмом. Для 8 задач число тактов: `ceil(8 / defaultParallelism)`.

## Задание 2. Генерация данных варианта 1

In [ ]:
SEED = 101
N_ROWS = 105_000
N_CAT = 5
N_BIG = 750_000
ROWS_PER_SEC = 10
WINDOW_SEC = 5
THRESHOLD = 55_000

def generate_events(seed, n_rows, n_cat):
    rs = np.random.RandomState(seed)
    regions = ['Астана', 'Алматы', 'Шымкент', 'Караганда', 'Актобе']
    cat_idx = rs.randint(0, n_cat, n_rows)
    reg_idx = rs.randint(0, 5, n_rows)
    amount = rs.randint(1000, 100001, n_rows)
    u = rs.rand(n_rows)
    sec = rs.randint(0, 7 * 24 * 3600, n_rows)
    base = datetime(2026, 9, 1)
    rows = []
    for i in range(n_rows):
        status = 'Выполнено' if u[i] < .80 else ('Отклонено' if u[i] < .90 else 'В обработке')
        moment = base + timedelta(seconds=int(sec[i]))
        rows.append((i+1, moment.strftime('%Y-%m-%d %H:%M:%S'),
                     'Категория ' + str(int(cat_idx[i])+1),
                     regions[int(reg_idx[i])], int(amount[i]), status))
    return rows

schema = StructType([
    StructField('event_id', LongType(), False),
    StructField('event_time', StringType(), False),
    StructField('category', StringType(), False),
    StructField('region', StringType(), False),
    StructField('amount', LongType(), False),
    StructField('status', StringType(), False)])
rows = generate_events(SEED, N_ROWS, N_CAT)
df = spark.createDataFrame(rows, schema).withColumn('event_time', F.to_timestamp('event_time'))
print("Строк:", df.count())
df.printSchema()
df.show(5, truncate=False)
print("Сумма amount:", sum(r[4] for r in rows))
ctrl = {}
for r in rows:
    c = ctrl.setdefault(r[2], [0, 0]); c[0] += 1; c[1] += r[4]
print("Категории [число записей, сумма]:")
for k in sorted(ctrl): print(k, ctrl[k])
statuses = {}
for r in rows: statuses[r[5]] = statuses.get(r[5], 0) + 1
print("Статусы:", statuses)

**Ответ к заданию 2.** Данные имитируют журнал заказов магазина одежды. Каждая запись содержит ID события, время, категорию, регион, сумму заказа и статус. Фиксированный `SEED=101` обеспечивает воспроизводимость. В отчёт перенеси число строк, общую сумму и статусы из вывода кода.

## Задание 3. RDD и сравнение с MapReduce

In [ ]:
rdd = df.rdd
print("Партиций RDD:", rdd.getNumPartitions())
t0 = time.perf_counter()
pairs = rdd.map(lambda r: (r.category, r.amount))
totals_rdd = pairs.reduceByKey(lambda a, b: a+b)
t_lazy = time.perf_counter() - t0
print("Время описания трансформаций: %.6f с" % t_lazy)
print("Граф lineage:")
print(totals_rdd.toDebugString().decode())
t0 = time.perf_counter()
totals = totals_rdd.collect()
t_run = time.perf_counter() - t0
print("Время collect(): %.6f с" % t_run)
print("Суммы:", sorted(totals))
print("Выполнено:", rdd.filter(lambda r: r.status == 'Выполнено').count())

def mr_sum(rows):
    pairs = [(r[2], r[4]) for r in rows]
    grouped = {}
    for k, v in pairs: grouped.setdefault(k, []).append(v)
    return {k: sum(vs) for k, vs in grouped.items()}
mr = mr_sum(rows)
print("RDD == MapReduce на Python:", dict(totals) == mr)

**Ответы к заданию 3.**
1. Описание трансформаций быстрое, потому что Spark строит план, но ещё не обрабатывает все строки. `collect()` и `count()` запускают вычисление.
2. В `toDebugString()` найди `ShuffledRDD`: shuffle связан с `reduceByKey`, это широкая трансформация.
3. Проверка `RDD == MapReduce на Python` должна быть `True`, что подтверждает совпадение сумм.
4. Spark самостоятельно планирует выполнение `map` и `reduceByKey`; в ручном MapReduce группировка и суммирование написаны явно.

## Задание 4. DataFrame API и Spark SQL

In [ ]:
by_cat = (df.groupBy('category')
          .agg(F.count('*').alias('n'), F.sum('amount').alias('total'),
               F.round(F.avg('amount'), 2).alias('avg_amount'))
          .orderBy('category'))
by_cat.show()
df.createOrReplaceTempView('events')
by_cat_sql = spark.sql('''
    SELECT category, COUNT(*) AS n, SUM(amount) AS total,
           ROUND(AVG(amount), 2) AS avg_amount
    FROM events GROUP BY category ORDER BY category
''')
by_cat_sql.show()
print("DataFrame API == SQL:", by_cat.collect() == by_cat_sql.collect())
pdf = pd.DataFrame(rows, columns=['event_id','event_time','category','region','amount','status'])
pd_tot = pdf.groupby('category')['amount'].sum().to_dict()
sp_tot = {r['category']: r['total'] for r in by_cat.collect()}
print("Spark == pandas:", sp_tot == pd_tot)

rej = spark.sql('''
    SELECT region,
    ROUND(100 * AVG(CASE WHEN status = 'Отклонено' THEN 1.0 ELSE 0.0 END), 2) AS rejected_pct
    FROM events GROUP BY region ORDER BY rejected_pct DESC
''')
print("Доля отклонённых по регионам:"); rej.show()
daily = spark.sql('''
    SELECT CAST(to_date(event_time) AS STRING) AS day, category,
           COUNT(*) AS n, SUM(amount) AS total
    FROM events GROUP BY to_date(event_time), category ORDER BY day, category
''')
print("По дням и категориям:"); daily.show(5)
above = df.filter(F.col('amount') > THRESHOLD).groupBy('category').count().orderBy('category')
print("События выше порога", THRESHOLD); above.show()
print("Физический план Catalyst:"); by_cat.explain()

**Ответы к заданию 4.**
1. Проверки `DataFrame API == SQL` и `Spark == pandas` должны вернуть `True`.
2. `HashAggregate` выполняет агрегацию; `Exchange` означает перераспределение данных (shuffle) для группировки.
3. Регион с максимальной долей отклонённых — первая строка `rej`; категория с максимальным количеством выше порога — строка `above` с наибольшим `count`.
4. RDD — коллекция без схемы; DataFrame — таблица со столбцами и оптимизацией Catalyst; Dataset — типизированная абстракция Scala/Java. В PySpark используют DataFrame. Для табличных запросов DataFrame обычно удобнее RDD.

## Задание 5. Сравнение времени и кэширование

In [ ]:
rs = np.random.RandomState(SEED)
k = rs.randint(0, N_CAT, N_BIG)
v = rs.randint(0, 1000, N_BIG)
pd.DataFrame({'k': k, 'v': v}).to_csv('big.csv', index=False)
print("Файл big.csv создан, строк:", N_BIG)

def py_agg():
    totals = {}
    with open('big.csv') as f:
        next(f)
        for line in f:
            a, b = line.split(',')
            a = int(a); totals[a] = totals.get(a, 0) + int(b)
    return totals
def pandas_agg():
    d = pd.read_csv('big.csv')
    return d.groupby('k')['v'].sum().to_dict()
sdf = spark.read.csv('big.csv', header=True, schema='k INT, v INT')
def spark_agg(d):
    result = d.groupBy('k').agg(F.sum('v').alias('total'))
    return {r['k']: r['total'] for r in result.collect()}
def timed(fn, *args):
    t0 = time.perf_counter(); res = fn(*args)
    return res, time.perf_counter() - t0

timings = []
r_py, t = timed(py_agg); timings.append(('Python (цикл по строкам)', t))
r_pd, t = timed(pandas_agg); timings.append(('pandas', t))
r_sp, t = timed(spark_agg, sdf); timings.append(('Spark, 1-й запуск', t))
rest = [timed(spark_agg, sdf)[1] for _ in range(3)]
timings.append(('Spark, повторные без cache (среднее)', sum(rest)/3))
sdf.cache(); sdf.count()
cached = [timed(spark_agg, sdf)[1] for _ in range(3)]
timings.append(('Spark, повторные с cache (среднее)', sum(cached)/3))
print("Результаты совпадают:", r_py == r_pd == r_sp)
timings_df = pd.DataFrame(timings, columns=['method', 'seconds'])
display(timings_df)
print("Без cache:", [round(x, 3) for x in rest])
print("С cache:", [round(x, 3) for x in cached])
if sum(cached) > 0:
    print("Отношение среднего без cache / с cache:",
          round((sum(rest)/len(rest))/(sum(cached)/len(cached)), 3), "раз")
sdf.unpersist()

**Ответы к заданию 5.**
1. Самый быстрый способ определи по фактической таблице. Spark может проигрывать pandas на локальных данных из-за накладных расходов.
2. Сравни среднее время без cache и с cache. Без кэша исходные данные читаются и вычисления выполняются заново.
3. `cache()` ленивый: кэш наполняется только после действия, здесь `sdf.count()`. Кэш полезен при многократном использовании одних данных.
4. Spark может держать промежуточные данные в памяти; Hadoop MapReduce традиционно записывает их на диск между этапами.
5. Повтори ячейку измерений второй раз и запиши оба набора результатов.

## Задание 6. Structured Streaming

In [ ]:
for q in spark.streams.active: q.stop()
cat_num = (F.col('value') % N_CAT + 1).cast('string')
stream = (spark.readStream.format('rate').option('rowsPerSecond', ROWS_PER_SEC).load()
          .withColumn('category', F.concat(F.lit('Категория '), cat_num)))
win = F.window('timestamp', str(WINDOW_SEC) + ' seconds')
agg = stream.groupBy(win, 'category').count()
query = (agg.writeStream.outputMode('complete').format('memory')
         .queryName('stream_counts').trigger(processingTime='2 seconds').start())
print("Поток запущен, активен:", query.isActive)

In [ ]:
time.sleep(22)
print("Последний микропакет:", query.lastProgress)
query.stop()
print("Поток остановлен, активен:", query.isActive)
res = spark.sql('''
    SELECT CAST(window.start AS STRING) AS window_start,
           CAST(window.end AS STRING) AS window_end, category, count AS cnt
    FROM stream_counts ORDER BY window_start, category
''')
res.show(40, truncate=False)
print("Ожидаемое число событий одной категории в полном окне:",
      ROWS_PER_SEC * WINDOW_SEC / N_CAT)

**Ответы к заданию 6.**
1. В варианте 1 ожидаемое число событий одной категории в полном окне: `10 × 5 ÷ 5 = 10`.
2. Первое и последнее окна неполные, потому что поток запускается и останавливается не обязательно на границе окна.
3. `complete` выводит всю результирующую таблицу; `memory` хранит её в памяти и подходит для отладки, но не для постоянного промышленного хранения.
4. Structured Streaming обрабатывает поступающие данные микропакетами; реальный источник может быть Apache Kafka вместо учебного `rate`.

## Задание 7. Экспорт CSV и Power BI

In [ ]:
def to_pdf(frame):
    return pd.DataFrame(frame.collect(), columns=frame.columns)
spark_part = to_pdf(by_cat)[['category','total']].rename(columns={'total':'total_spark'})
py_part = pd.DataFrame(sorted(mr.items()), columns=['category','total_python'])
cmp_df = spark_part.merge(py_part, on='category')
cmp_df['match'] = cmp_df['total_spark'] == cmp_df['total_python']
enc = 'utf-8-sig'
cmp_df.to_csv('spark_totals.csv', index=False, encoding=enc)
to_pdf(daily).to_csv('spark_daily.csv', index=False, encoding=enc)
to_pdf(res).to_csv('stream_windows.csv', index=False, encoding=enc)
timings_df.to_csv('timings.csv', index=False, encoding=enc)
display(cmp_df)
print("Созданы: spark_totals.csv, spark_daily.csv, stream_windows.csv, timings.csv")

In [ ]:
try:
    from google.colab import files
    for name in ['spark_totals.csv','spark_daily.csv','stream_windows.csv','timings.csv']:
        files.download(name)
except ImportError:
    print("CSV сохранены в:", os.getcwd())

### Инструкция Power BI
1. **Главная → Получить данные → Текст/CSV** и импортируй четыре CSV-файла.
2. Проверь типы: `window_start`, `window_end` — дата/время; `day` — дата; суммы и счётчики — целые числа; `seconds` — десятичное; `match` — True/False.
3. Создай страницу **«Сравнение результатов Spark»** и не менее 6 визуализаций:
   - кластерная столбчатая диаграмма `total_spark` и `total_python` по `category`;
   - карточка общей суммы;
   - карточка количества категорий, где `match=True`;
   - линейчатая диаграмма `seconds` по `method`, сортировка по убыванию;
   - график `cnt` по `window_start` с легендой `category`;
   - диаграмма сумм по дням из `spark_daily`;
   - два среза: по категории и по дню.
4. Используй информативные заголовки, подписи осей, ограниченную палитру и не используй 3D-диаграммы.
5. Добавь четыре вывода: совпали ли Spark и Python; какой способ быстрее; как повлиял cache; почему крайние потоковые окна неполные.

## Задание 8. Аналитическая записка

Apache Spark — движок распределённой обработки данных, который позволяет сохранять промежуточные результаты в оперативной памяти. В этой работе Spark запущен в локальном режиме Google Colab. Драйвер координирует программу, менеджер кластера выделяет ресурсы, исполнители выполняют задачи над партициями. Job запускается действием и делится на stages по границам shuffle; каждая stage состоит из tasks.

RDD — неизменяемая распределённая коллекция без табличной схемы, подходящая для нестандартных преобразований. DataFrame — распределённая таблица с именованными столбцами и типами, удобная для аналитических запросов. Dataset поддерживает проверку типов на этапе компиляции в Scala и Java; в PySpark используется DataFrame. Catalyst оптимизирует планы DataFrame и Spark SQL.

Spark SQL подходит для отчётов интернет-магазина: суммы заказов по категориям и дням, средний заказ и доля отклонённых операций по регионам. Structured Streaming можно использовать для агрегации потока заказов по временным окнам; реальным источником событий может служить Kafka. MLlib может применяться для прогноза спроса или поиска необычных заказов.

| Критерий | Apache Spark | Hadoop MapReduce |
|---|---|---|
| Промежуточные данные | Могут храниться в памяти через cache/persist | Обычно записываются на диск между этапами |
| Повторное использование | Кэширование и итеративные вычисления | Часто повторное чтение данных |
| Задержка | Поддерживает микропакетную потоковую обработку | Преимущественно пакетная обработка |
| Интерфейсы | RDD, DataFrame, SQL, MLlib, GraphX | Низкоуровневые Map/Reduce |
| Требования к памяти | Может требовать много RAM | Сильнее опирается на диск |
| Хранение | Собственной файловой системы нет; использует внешние источники | Часто использует HDFS |

В окончательный отчёт внеси фактические значения времени из `timings_df`. Первый запуск Spark включает накладные расходы, а кэширование может ускорить повторные операции. На небольших данных pandas или чистый Python могут оказаться быстрее. Spark оправдан при больших объёмах, повторном использовании данных, SQL-аналитике и потоковой обработке; ограничения — требования к памяти и накладные расходы запуска и обмена данными.

## Контрольные вопросы — ответы

1. **Apache Spark** — универсальный движок распределённой обработки больших данных с возможностью вычислений в памяти.
2. Основные модули: Spark Core, Spark SQL, Structured Streaming, MLlib, GraphX.
3. **RDD** — неизменяемая распределённая коллекция, разделённая на партиции и восстанавливаемая по lineage.
4. **DataFrame** — таблица с именованными столбцами; **Dataset** — типизированная абстракция Scala/Java. В PySpark используется DataFrame.
5. Преимущества: кэширование, развитые API, оптимизация SQL, потоковая обработка, MLlib.
6. Spark может хранить промежуточные данные в памяти; MapReduce обычно записывает их на диск.
7. Spark часто быстрее при повторных вычислениях, но не обязан быть быстрее на небольших локальных данных.
8. Языки: Scala, Java, Python, R и SQL.
9. Применение: ETL, аналитика, потоки событий, машинное обучение, анализ графов.
10. Трансформации лениво строят граф; действия запускают вычисление. Граф преобразований называется lineage.
11. Партиция — часть набора данных и единица параллелизма; параллельность ограничена доступными ядрами и ресурсами.
12. Structured Streaming обрабатывает данные микропакетами; оконная агрегация группирует события по интервалам времени.

## Тест по теме 6 — правильные ответы

1. **Б** — универсальный движок распределённой обработки больших данных с вычислениями в оперативной памяти.
2. **Д** — HDFS не является модулем Spark; это файловая система экосистемы Hadoop.
3. **А** — неизменяемая распределённая коллекция элементов, разбитая на партиции и восстанавливаемая по lineage.
4. **Г** — `collect()` является действием, запускающим вычисление.
5. **В** — DataFrame имеет схему с именованными столбцами и типами, поэтому Catalyst может оптимизировать план.
6. **Б** — типизированный Dataset API доступен в Scala и Java.
7. **Г** — Spark может хранить промежуточные данные в оперативной памяти вместо записи на диск между этапами.
8. **В** — кэш наполняется при первом действии над DataFrame.
9. **А** — `reduceByKey` частично объединяет значения до shuffle, уменьшая объём передаваемых данных.
10. **Д** — 4 задачи: два исполнителя × два ядра на исполнителя = 4 задачи одновременно.

## Вывод

В работе запущена сессия Spark, сгенерирован набор событий интернет-магазина одежды, выполнены агрегации средствами RDD, DataFrame API и Spark SQL. Результаты сопоставлены с Python и pandas. Измерено время обработки и влияние кэширования, реализована оконная агрегация Structured Streaming, подготовлены CSV-файлы для Power BI. Spark полезен для распределённой, повторной и потоковой обработки данных, но фактическое ускорение зависит от объёма данных и ресурсов среды.